In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import statsmodels.api as sm

import concurrent.futures


import numpy as np
import pandas as pd
import ast
import glob
import pickle
import dask
import os
import itertools

import pickle

#from sklearn.cluster import KMeans
from sklearn.metrics import pairwise_distances, mean_squared_error, mean_absolute_error
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split, cross_val_score
from statsmodels.regression.rolling import RollingOLS

from tqdm.notebook import tqdm

from multiprocessing import Pool, cpu_count
from joblib import Parallel, delayed
#from tqdm import tqdm
from collections import Counter
from functools import reduce


#import dask
#import dask.dataframe as dd
#from dask.distributed import Client
#from dask.diagnostics import ProgressBar

#client = Client(n_workers=20, memory_limit="10GB", interface='lo')
from concurrent.futures import ThreadPoolExecutor

#import dask_ml.cluster as dask_cluster

from pprint import pprint
import os

pd.set_option('display.max_columns', None)

import warnings
warnings.filterwarnings('ignore', category=FutureWarning)

### Load Augmented DF

In [ ]:
augmented_df = pd.read_parquet("../data/augmented_us_counties.parquet")
augmented_df["date"] = pd.to_datetime(augmented_df["date"])
augmented_df["fips"] = augmented_df["fips"].astype(int)
augmented_df["days_from_start"] = augmented_df["days_from_start"].astype(int)
augmented_df = augmented_df.sort_values(by=["fips","date"])


In [ ]:
# Check for gaps
gt_columns = ["fips", "days_from_start"]
augmented_df_gt = augmented_df[gt_columns]
grouped = augmented_df_gt.groupby('fips')

for fips, group in grouped:
    missing_days = group['days_from_start'].diff().gt(1).sum()
    if missing_days > 0:
        print(f"Gap(s) found in 'days_from_start' for fips {fips}: {missing_days} gap(s)")


### Load TLGRF Benchmark Dataset

In [ ]:
benchmark_TLGRF_dataset = pd.read_parquet("../data/benchmark_tlgrf/")
benchmark_TLGRF_dataset["date"] = pd.to_datetime(benchmark_TLGRF_dataset["date"])

df = benchmark_TLGRF_dataset.copy()
fips_list = df["fips"].unique()
display(benchmark_TLGRF_dataset)


### Define `read_csv_file` helper

In [ ]:
def read_csv_file(file_path):
    # Read the CSV file into a pandas DataFrame
    try:
        GRF_df = pd.read_csv(file_path)
        return GRF_df
    except pd.errors.EmptyDataError:
        print(file_path)

### Read Time Variant no CUSP GRF

In [ ]:
# time_variant_no_CUSP variant not in final paper — skipped
time_variant_no_CUSP_GRF_dfs = []


In [ ]:
# Empty placeholder — no_CUSP not in final paper
time_variant_no_CUSP_GRF_results = pd.DataFrame(
    columns=["fips","days_from_start","date",
             "GRF_predicted_log_rolled_cases",
             "log_rolled_cases","shifted_log_rolled_cases"])
time_variant_no_CUSP_GRF_results["date"] = pd.to_datetime(
    time_variant_no_CUSP_GRF_results["date"])


### Read Time Variant GRF

In [ ]:
time_variant_GRF_dfs = [pd.read_parquet("../data/benchmark_grf_time_variant/")]


In [ ]:
time_variant_GRF_results = pd.concat(time_variant_GRF_dfs).sort_values(by=["fips", "days_from_start"])
time_variant_GRF_results["date"] = pd.to_datetime(time_variant_GRF_results["date"])
time_variant_GRF_results = time_variant_GRF_results.rename(columns={"predicted_log_rolled_cases_GRF":"GRF_predicted_log_rolled_cases"})
time_variant_GRF_results = time_variant_GRF_results.dropna(subset=["GRF_predicted_log_rolled_cases"])
time_variant_GRF_results = pd.merge(time_variant_GRF_results, augmented_df_gt, on=["fips","days_from_start"], how="left")
#time_invariant_GRF_results = time_invariant_GRF_results.drop(columns=["predicted.grf.future", "predicted.grf.future.0", "Predicted_Double_Days"])
time_variant_GRF_results.to_csv("time_variant_GRF_results_df.csv", index=False)

display(time_variant_GRF_results)

In [ ]:
shifted_W_time_variant_GRF_dfs = [pd.read_parquet("../data/benchmark_grf_shifted_W_time_variant/")]


In [ ]:
shifted_W_time_variant_GRF_results = pd.concat(shifted_W_time_variant_GRF_dfs).sort_values(by=["fips", "days_from_start"])
shifted_W_time_variant_GRF_results["date"] = pd.to_datetime(shifted_W_time_variant_GRF_results["date"])
shifted_W_time_variant_GRF_results = shifted_W_time_variant_GRF_results.rename(columns={"predicted_log_rolled_cases_GRF":"GRF_predicted_log_rolled_cases"})
shifted_W_time_variant_GRF_results = shifted_W_time_variant_GRF_results.dropna(subset=["GRF_predicted_log_rolled_cases"])
shifted_W_time_variant_GRF_results = pd.merge(shifted_W_time_variant_GRF_results, augmented_df_gt, on=["fips","days_from_start"], how="left")
#time_invariant_GRF_results = time_invariant_GRF_results.drop(columns=["predicted.grf.future", "predicted.grf.future.0", "Predicted_Double_Days"])

shifted_W_time_variant_GRF_results.to_csv("shifted_W_time_variant_GRF_results_df.csv", index=False)

display(shifted_W_time_variant_GRF_results)

### Read Time Invariant GRF

In [ ]:
time_invariant_GRF_dfs = [pd.read_parquet("../data/benchmark_grf_time_invariant/")]


In [ ]:
time_invariant_GRF_results = pd.concat(time_invariant_GRF_dfs).sort_values(by=["fips", "days_from_start"])
time_invariant_GRF_results["date"] = pd.to_datetime(time_invariant_GRF_results["date"])
time_invariant_GRF_results = time_invariant_GRF_results.rename(columns={"predicted_log_rolled_cases_GRF":"GRF_predicted_log_rolled_cases"})
time_invariant_GRF_results = time_invariant_GRF_results.dropna(subset=["GRF_predicted_log_rolled_cases"])

time_invariant_GRF_results = pd.merge(time_invariant_GRF_results, augmented_df_gt, on=["fips","days_from_start"], how="left")
#time_invariant_GRF_results = time_invariant_GRF_results.drop(columns=["predicted.grf.future", "predicted.grf.future.0", "Predicted_Double_Days"])

time_invariant_GRF_results.to_csv("time_invariant_GRF_results_df.csv",index=False)

display(time_invariant_GRF_results)

In [ ]:
shifted_W_time_invariant_GRF_dfs = [pd.read_parquet("../data/benchmark_grf_shifted_W_time_invariant/")]


In [ ]:
shifted_W_time_invariant_GRF_results = pd.concat(shifted_W_time_invariant_GRF_dfs).sort_values(by=["fips", "days_from_start"])
shifted_W_time_invariant_GRF_results["date"] = pd.to_datetime(shifted_W_time_invariant_GRF_results["date"])
shifted_W_time_invariant_GRF_results = shifted_W_time_invariant_GRF_results.rename(columns={"predicted_log_rolled_cases_GRF":"GRF_predicted_log_rolled_cases"})
shifted_W_time_invariant_GRF_results = shifted_W_time_invariant_GRF_results.dropna(subset=["GRF_predicted_log_rolled_cases"])

shifted_W_time_invariant_GRF_results = pd.merge(shifted_W_time_invariant_GRF_results, augmented_df_gt, on=["fips","days_from_start"], how="left")
#time_invariant_GRF_results = time_invariant_GRF_results.drop(columns=["predicted.grf.future", "predicted.grf.future.0", "Predicted_Double_Days"])
shifted_W_time_invariant_GRF_results.to_csv("shifted_W_time_invariant_GRF_results_df.csv", index=False)
display(shifted_W_time_invariant_GRF_results)


### Compute RMSE and MAE

In [ ]:
rmse_GRF_func = lambda x: np.sqrt(np.nanmean((x["GRF_predicted_log_rolled_cases"] - x["shifted_log_rolled_cases"]) ** 2))
mae_GRF_func = lambda x: np.nanmean(np.abs(x["GRF_predicted_log_rolled_cases"] - x["shifted_log_rolled_cases"]))
rmse_TLGRF_func = lambda x: np.sqrt(np.nanmean((x["TLGRF_predicted_log_rolled_cases"] - x["shifted_log_rolled_cases"]) ** 2))
mae_TLGRF_func = lambda x: np.nanmean(np.abs(x["TLGRF_predicted_log_rolled_cases"] - x["shifted_log_rolled_cases"]))
#

log_20 = np.log(20 + 1.1)

TLGRF_df = benchmark_TLGRF_dataset[benchmark_TLGRF_dataset["log_rolled_cases"] >= log_20]
TLGRF_df = TLGRF_df[TLGRF_df["date"] <= "2022-12-31"]


time_invariant_GRF_df = time_invariant_GRF_results[time_invariant_GRF_results["log_rolled_cases"] >= log_20]
time_invariant_GRF_df = time_invariant_GRF_df[time_invariant_GRF_df["date"] <= "2022-12-31"]

time_variant_GRF_df = time_variant_GRF_results[time_variant_GRF_results["log_rolled_cases"] >= log_20]
time_variant_GRF_df = time_variant_GRF_df[time_variant_GRF_df["date"] <= "2022-12-31"]




shifted_W_time_invariant_GRF_df = shifted_W_time_invariant_GRF_results[shifted_W_time_invariant_GRF_results["log_rolled_cases"] >= log_20]
shifted_W_time_invariant_GRF_df = shifted_W_time_invariant_GRF_df[shifted_W_time_invariant_GRF_df["date"] <= "2022-12-31"]

shifted_W_time_variant_GRF_df = shifted_W_time_variant_GRF_results[shifted_W_time_variant_GRF_results["log_rolled_cases"] >= log_20]
shifted_W_time_variant_GRF_df = shifted_W_time_variant_GRF_df[shifted_W_time_variant_GRF_df["date"] <= "2022-12-31"]



#TLGRF
RMSE_TLGRF = TLGRF_df.groupby("date").apply(rmse_TLGRF_func)
MAE_TLGRF = TLGRF_df.groupby("date").apply(mae_TLGRF_func)
#Time Invariant GRF
RMSE_time_invariant_GRF_df = time_invariant_GRF_df.groupby("date").apply(rmse_GRF_func)
MAE_time_invariant_GRF_df = time_invariant_GRF_df.groupby("date").apply(mae_GRF_func)
#Time Variant GRF
RMSE_time_variant_GRF_df = time_variant_GRF_df.groupby("date").apply(rmse_GRF_func)
MAE_time_variant_GRF_df = time_variant_GRF_df.groupby("date").apply(mae_GRF_func)
#Time Variant no CUSP GRF



# Shifted W Time Invariant
RMSE_shifted_W_time_invariant_GRF_df = shifted_W_time_invariant_GRF_df.groupby("date").apply(rmse_GRF_func)
MAE_shifted_W_time_invariant_GRF_df = shifted_W_time_invariant_GRF_df.groupby("date").apply(mae_GRF_func)
# Shifted W Time Variant GRF
RMSE_shifted_W_time_variant_GRF_df = shifted_W_time_variant_GRF_df.groupby("date").apply(rmse_GRF_func)
MAE_shifted_W_time_variant_GRF_df = shifted_W_time_variant_GRF_df.groupby("date").apply(mae_GRF_func)



In [ ]:
combined_daily_metrics_df = pd.DataFrame()
combined_daily_metrics_df["MAE TLGRF"] = MAE_TLGRF
combined_daily_metrics_df["RMSE TLGRF"] = RMSE_TLGRF
combined_daily_metrics_df["MAE Time Invariant GRF"] = MAE_time_invariant_GRF_df
combined_daily_metrics_df["RMSE Time Invariant GRF"] = RMSE_time_invariant_GRF_df
combined_daily_metrics_df["MAE Time Variant GRF"] = MAE_shifted_W_time_variant_GRF_df
combined_daily_metrics_df["RMSE Time Variant GRF"] = RMSE_shifted_W_time_variant_GRF_df


combined_daily_metrics_df.to_csv("combined_GRF_daily_metrics.csv" ,index=True)

In [ ]:
combined_daily_metrics_df

In [ ]:
metrics_comparison_df = pd.DataFrame()

metrics_comparison_df["MAE"] = [MAE_time_invariant_GRF_df.median(), MAE_shifted_W_time_variant_GRF_df.median(), MAE_TLGRF.median()]
metrics_comparison_df["RMSE"] = [RMSE_time_invariant_GRF_df.median(), RMSE_shifted_W_time_variant_GRF_df.median(), RMSE_TLGRF.median()]
metrics_comparison_df.index = ["Time Invariant GRF", "Time Variant GRF", "TLGRF"]
metrics_comparison_df

In [ ]:
[RMSE_time_invariant_GRF_df.median(), RMSE_TLGRF.median()]

In [ ]:
[MAE_time_invariant_GRF_df.median(), MAE_TLGRF.median()]

In [ ]:
days_to_date_df = time_variant_GRF_results[["days_from_start","date"]]
days_to_date_df.drop_duplicates(inplace=True)
days_to_date_df = days_to_date_df.sort_values(by="date")

date0 = days_to_date_df["date"].min() - pd.Timedelta(days=days_to_date_df["days_from_start"].min())
date0 = date0.date()
missing_days_df = pd.DataFrame({'days_from_start': range(0, days_to_date_df["days_from_start"].max()),
                                'date': pd.date_range(start=date0, periods=days_to_date_df["days_from_start"].max())})
#date0, days_to_date_df["date"].min(), pd.Timedelta(days=days_to_date_df["days_from_start"].min())
missing_days_df

In [ ]:
days_to_date_df[days_to_date_df["date"]=="2021-09-12"]

### Plot TLGRF vs GRF

In [ ]:
plt.figure(figsize=(12,6))

#plt.plot(tcv_performance_df["rmse"], label="tcv")
#plt.plot(ctcv_performance_df["rmse"], label="ctcv")
#plt.plot(RMSE_shifted_W_time_invariant_GRF_df, label="Time Invariant Shifted W GRF")
#plt.plot(RMSE_shifted_W_time_variant_GRF_df, label="Time Variant Shifted W GRF")
plt.plot(RMSE_time_invariant_GRF_df, label="Direct causal_forest() Implementation\nwithout time varying features")
plt.plot(RMSE_shifted_W_time_variant_GRF_df, label="Direct causal_forest() Implementation\nwith time varying features")
##plt.plot(RMSE_time_variant_no_CUSP_GRF_df, label="Time Variant no CUSP GRF")
plt.plot(RMSE_TLGRF, label="TLGRF", color="red")

plt.legend(loc='upper right', fontsize='large')  # Adjust the bbox_to_anchor to increase the size
plt.xlabel("Date")
plt.ylabel("RMSE")
plt.title("Root Mean Square Error (RMSE) in One-Week Ahead COVID Case Predictions")
plt.xlim(pd.to_datetime('2020-03-15'), pd.to_datetime('2023-01-01'))
plt.ylim(0,1.0)
plt.savefig("variant_and_invariant_grf_tlgrf_rmse.png")
plt.show()


In [ ]:
plt.figure(figsize=(20,10))

plt.plot(RMSE_shifted_W_time_variant_GRF_df, label="Time Variant GRF")
## plt.plot(RMSE_time_variant_no_CUSP_GRF_df, label="Time Variant no CUSP GRF")
#plt.plot(RMSE_TLGRF, label="TLGRF", color="red")

plt.legend()
plt.xlabel("Date")
plt.ylabel("RMSE")
plt.title("Root Mean Square Error (RMSE) in One-Week Ahead COVID Case Predictions")
plt.xlim(pd.to_datetime('2020-03-15'), pd.to_datetime('2023-01-01'))
plt.ylim(0,1.0)
plt.savefig("GRF_time_variant_cusp_vs_no_cusp_rmse.png")
plt.show()


In [ ]:
plt.figure(figsize=(20,10))

plt.plot(RMSE_shifted_W_time_variant_GRF_df, label="Time Variant GRF")
##plt.plot(RMSE_time_variant_no_CUSP_GRF_df, label="Time Variant no CUSP GRF")
plt.plot(RMSE_TLGRF, label="TLGRF", color="red")

plt.legend()
plt.xlabel("Date")
plt.ylabel("RMSE")
plt.title("Root Mean Square Error (RMSE) in One-Week Ahead COVID Case Predictions")
plt.xlim(pd.to_datetime('2020-03-15'), pd.to_datetime('2023-01-01'))
plt.ylim(0,1.0)
plt.savefig("GRF_time_variant_vs_TLGRF_rmse.png")
plt.show()


In [ ]:
plt.figure(figsize=(20,10))

plt.plot(RMSE_shifted_W_time_variant_GRF_df, label="Time Variant GRF")
## plt.plot(RMSE_time_variant_no_CUSP_GRF_df, label="Time Variant no CUSP GRF")
plt.plot(RMSE_TLGRF, label="TLGRF", color="red")

plt.legend()
plt.xlabel("Date")
plt.ylabel("RMSE")
plt.title("Root Mean Square Error (RMSE) in One-Week Ahead COVID Case Predictions")
plt.xlim(pd.to_datetime('2020-03-15'), pd.to_datetime('2023-01-01'))
plt.ylim(0,1.0)
# plt.savefig("GRF_time_variant_CUSP_vs_no_CUSP_vs_TLGRF_rmse.png")
plt.show()

In [ ]:
plt.figure(figsize=(20,10))

#plt.plot(tcv_performance_df["rmse"], label="tcv")
#plt.plot(ctcv_performance_df["rmse"], label="ctcv")
#plt.plot(RMSE_shifted_W_time_invariant_GRF_df, label="Time Invariant Shifted W GRF")
#plt.plot(RMSE_shifted_W_time_variant_GRF_df, label="Time Variant Shifted W GRF")
plt.plot(RMSE_time_invariant_GRF_df, label="Classical GRF", linestyle="dashed")
#plt.plot(RMSE_time_variant_GRF_df, label="Time Variant GRF")
plt.plot(RMSE_TLGRF, label="TLGRF", color="red")

plt.legend()
plt.xlabel("Date")
plt.ylabel("RMSE")
plt.title("Root Mean Square Error (RMSE) in One-Week Ahead COVID Case Predictions")
plt.xlim(pd.to_datetime('2020-03-15'), pd.to_datetime('2023-01-01'))
plt.ylim(0,1.0)
plt.savefig("updated_grf_tlgrf_rmse.png")
plt.show()


### MAE

In [ ]:
plt.figure(figsize=(16,8))

plt.plot(MAE_time_invariant_GRF_df, label="Direct causal_forest() Implementation\nwithout time varying features")
plt.plot(MAE_shifted_W_time_variant_GRF_df, label="Direct causal_forest() Implementation\nwith time varying features")
##plt.plot(MAE_time_variant_no_CUSP_GRF_df, label="Time Variant GRF")

plt.plot(MAE_TLGRF, label="TLGRF", color="r")

plt.legend(loc='upper right', fontsize='large')  # Adjust the bbox_to_anchor to increase the size
plt.xlabel("Date")
plt.ylabel("MAE")
plt.title("Mean Absolute Error (MAE) in One-Week Ahead COVID Case Predictions")
plt.xlim(pd.to_datetime('2020-03-15'), pd.to_datetime('2023-01-01'))
plt.ylim(0,0.8)
plt.savefig("variant_and_invariant_grf_tlgrf_mae.png")

plt.show()

In [ ]:
plt.figure(figsize=(20,10))

#plt.plot(tcv_performance_df["mae"], label="tcv")
#plt.plot(ctcv_performance_df["mae"], label="ctcv")
#plt.plot(MAE_shifted_W_time_invariant_GRF_df, label="Time Invariant Shifted W GRF")
#plt.plot(MAE_shifted_W_time_variant_GRF_df, label="Time Variant Shifted W GRF")
plt.plot(MAE_shifted_W_time_variant_GRF_df, label="Time Variant GRF")
#plt.plot(MAE_time_variant_no_CUSP_GRF_df, label="Time Variant no CUSP GRF")

plt.legend()
plt.xlabel("Date")
plt.ylabel("MAE")
plt.title("Mean Absolute Error (MAE) in One-Week Ahead COVID Case Predictions")
plt.xlim(pd.to_datetime('2020-03-15'), pd.to_datetime('2023-01-01'))
plt.ylim(0,0.6)
plt.savefig("GRF_time_variant_cusp_vs_no_cusp_mae.png")

plt.show()

In [ ]:
plt.figure(figsize=(20,10))

plt.plot(MAE_shifted_W_time_variant_GRF_df, label="Time Variant GRF")
#plt.plot(MAE_time_variant_no_CUSP_GRF_df, label="Time Variant no CUSP GRF")
plt.plot(MAE_TLGRF, label="TLGRF", color="red")

plt.legend()
plt.xlabel("Date")
plt.ylabel("MAE")
plt.title("Mean Absolute Error (MAE) in One-Week Ahead COVID Case Predictions")
plt.xlim(pd.to_datetime('2020-03-15'), pd.to_datetime('2023-01-01'))
plt.ylim(0,0.6)
plt.savefig("GRF_time_variant_CUSP_vs_no_CUSP_vs_TLGRF_mae.png")
plt.show()

In [ ]:
plt.figure(figsize=(20,10))

#plt.plot(tcv_performance_df["rmse"], label="tcv")
#plt.plot(ctcv_performance_df["rmse"], label="ctcv")
#plt.plot(RMSE_shifted_W_time_invariant_GRF_df, label="Time Invariant Shifted W GRF")
#plt.plot(RMSE_shifted_W_time_variant_GRF_df, label="Time Variant Shifted W GRF")
plt.plot(MAE_time_invariant_GRF_df, label="Classical GRF", linestyle="dashed")
#plt.plot(RMSE_time_variant_GRF_df, label="Time Variant GRF")
plt.plot(MAE_TLGRF, label="TLGRF", color="red")

plt.legend()
plt.xlabel("Date")
plt.ylabel("MAE")
plt.title("Mean Absolute Error (MAE) in One-Week Ahead COVID Case Predictions")
plt.xlim(pd.to_datetime('2020-03-15'), pd.to_datetime('2023-01-01'))
plt.ylim(0,0.6)
plt.savefig("updated_grf_tlgrf_mae.png")
plt.show()

In [ ]:
def check_gt(df1, df2):
    #print(df1.shape)
    #print(df2.shape)
    check_df = pd.merge(df1, df2, on=["fips","days_from_start"], how="inner")
    check_x = np.sum(check_df["log_rolled_cases_x"] - check_df["log_rolled_cases_y"])
    check_y = np.sum(check_df["shifted_log_rolled_cases_x"] - check_df["shifted_log_rolled_cases_y"])
    return (check_df, check_x, check_y)

In [ ]:
RMSE_TLGRF.to_csv("RMSE_TLGRF.csv")
MAE_TLGRF.to_csv("MAE_TLGRF.csv")
RMSE_time_invariant_GRF_df.to_csv("RMSE_time_invariant_GRF_df.csv")
MAE_time_invariant_GRF_df.to_csv("MAE_time_invariant_GRF_df.csv")
RMSE_time_variant_GRF_df.to_csv("RMSE_time_variant_GRF_df.csv")
MAE_time_variant_GRF_df.to_csv("MAE_time_variant_GRF_df.csv")


RMSE_shifted_W_time_invariant_GRF_df.to_csv("RMSE_shifted_W_time_invariant_GRF_df.csv")
MAE_shifted_W_time_invariant_GRF_df.to_csv("MAE_shifted_W_time_invariant_GRF_df.csv")
RMSE_shifted_W_time_variant_GRF_df.to_csv("RMSE_shifted_W_time_variant_GRF_df.csv")
MAE_shifted_W_time_variant_GRF_df.to_csv("MAE_shifted_W_time_variant_GRF_df.csv")